In [5]:
!pip install opencv-python
!pip install mediapipe
!pip install scikit-learn
!pip install pandas

  Using cached numpy-2.2.6-cp311-cp311-win_amd64.whl.metadata (60 kB)
Using cached numpy-2.2.6-cp311-cp311-win_amd64.whl (12.9 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mediapipe 0.10.21 requires numpy<2, but you have numpy 2.2.6 which is incompatible.

[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached numpy-1.26.4-cp311-cp311-win_amd64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp311-cp311-win_amd64.whl (15.8 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.2.6
    Uninstalling numpy-2.2.6:
      Successfully uninstalled numpy-2.2.6


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.

[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import cv2
import mediapipe as mp
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, accuracy_score
from collections import defaultdict, Counter, deque
from scipy.spatial.distance import cdist
from collections import Counter
import os

In [9]:
import os
import csv
import cv2
import numpy as np
import mediapipe as mp
from sklearn.model_selection import train_test_split

# === Funzione per estrarre keypoints da un percorso e salvarli in CSV ===
def extract_keypoints(dataset_path, csv_file):
    file_list = {}

    for cls in os.listdir(dataset_path):
        cls_path = os.path.join(dataset_path, cls)
        if os.path.isdir(cls_path):
            images = [os.path.join(cls_path, f) for f in os.listdir(cls_path)]
            file_list[cls] = images

    # Header CSV: class + 21 keypoints (x,y)
    landmarks = ['class']
    for val in range(1, 22):
        landmarks += ['x{}'.format(val), 'y{}'.format(val)]

    with open(csv_file, mode='w', newline='') as f:
        csv_writer = csv.writer(f, delimiter=',', quotechar='"', quoting=csv.QUOTE_MINIMAL)
        csv_writer.writerow(landmarks)

    mp_hands = mp.solutions.hands
    with mp_hands.Hands(static_image_mode=True, max_num_hands=1, min_detection_confidence=0.5) as hands:
        for cls, files in file_list.items():
            print(f"Processing class: {cls}")
            for file in files:
                image = cv2.imread(file)
                if image is None:
                    continue
                image = cv2.flip(image, 1)
                results = hands.process(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))

                try:
                    for hand_landmark in results.multi_hand_landmarks:
                        keypoints = list(np.array([[lm.x, lm.y] for lm in hand_landmark.landmark]).flatten())
                        row = [cls] + keypoints
                        with open(csv_file, mode='a', newline='') as f:
                            csv_writer = csv.writer(f, delimiter=',', quotechar='"', quoting=csv.QUOTE_MINIMAL)
                            csv_writer.writerow(row)
                except:
                    pass

In [10]:
# === Percorsi dataset ===
train_path = r"C:\Users\nicol\Desktop\Progetti\Machine learning\project\asl_alphabet_train"

# === Estrai keypoints per train e test separati ===
extract_keypoints(train_path, "hand_dataset_train.csv")


Processing class: A
Processing class: B
Processing class: C
Processing class: D
Processing class: E
Processing class: F
Processing class: G
Processing class: H
Processing class: I
Processing class: J
Processing class: K
Processing class: L
Processing class: M
Processing class: N
Processing class: O
Processing class: P
Processing class: Q
Processing class: R
Processing class: S
Processing class: T
Processing class: U
Processing class: V
Processing class: W
Processing class: X
Processing class: Y
Processing class: Z


In [11]:
test_path = r"C:\Users\nicol\Desktop\Progetti\Machine learning\project\asl_alphabet_test"
extract_keypoints(test_path, "hand_dataset_test.csv")

Processing class: A
Processing class: B
Processing class: C
Processing class: D
Processing class: E
Processing class: F
Processing class: G
Processing class: H
Processing class: I
Processing class: J
Processing class: K
Processing class: L
Processing class: M
Processing class: N
Processing class: O
Processing class: P
Processing class: Q
Processing class: R
Processing class: S
Processing class: T
Processing class: U
Processing class: V
Processing class: W
Processing class: X
Processing class: Y
Processing class: Z


In [12]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ===============================
# 1) CARICA TRAIN DATASET
# ===============================
train_dataset = pd.read_csv("hand_dataset_train.csv")
X = train_dataset.iloc[:, 1:].values
Y = train_dataset.iloc[:, 0].values

# Splitta in train e validation (20% validation)
X_train, X_val, y_train, y_val = train_test_split(
    X, Y, test_size=0.2, random_state=42, stratify=Y
)

# ===============================
# 2) CARICA TEST DATASET
# ===============================
test_dataset = pd.read_csv("hand_dataset_test.csv")
X_test = test_dataset.iloc[:, 1:].values
y_test = test_dataset.iloc[:, 0].values

# ===============================
# 3) SCALING
# ===============================
scaler = StandardScaler().fit(X_train)

X_train = scaler.transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

# ===============================
# 4) INFO SHAPE
# ===============================
print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)


X_train: (21576, 42)
X_val: (5394, 42)
X_test: (2952, 42)


In [13]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report
import numpy as np

# Lista dei valori di k da provare
k_values = list(range(1, 21))  # prova k da 1 a 20
best_k = 1
best_val_acc = 0

# ----- Ricerca del miglior k sul validation set -----
for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)             # train sul training set
    y_val_pred = knn.predict(X_val)       # predizione sul validation set
    val_acc = accuracy_score(y_val, y_val_pred)
    
    print(f"k={k}, Validation Accuracy={val_acc:.4f}")
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_k = k

print(f"\nMiglior k trovato: {best_k} con accuracy validation={best_val_acc:.4f}")

# ----- Riaddestra sul training + validation set -----
X_train_full = np.vstack((X_train, X_val))
y_train_full = np.concatenate((y_train, y_val))

final_knn = KNeighborsClassifier(n_neighbors=best_k)
final_knn.fit(X_train_full, y_train_full)

# ----- Valutazione sul test set -----
y_test_pred = final_knn.predict(X_test)
test_acc = accuracy_score(y_test, y_test_pred)

print(f"Test Accuracy con k={best_k}: {test_acc:.4f}")
print("\nClassification report (test set):")
print(classification_report(y_test, y_test_pred))



k=1, Validation Accuracy=0.9359
k=2, Validation Accuracy=0.9164
k=3, Validation Accuracy=0.9166
k=4, Validation Accuracy=0.9067
k=5, Validation Accuracy=0.9030
k=6, Validation Accuracy=0.8978
k=7, Validation Accuracy=0.8893
k=8, Validation Accuracy=0.8891
k=9, Validation Accuracy=0.8860
k=10, Validation Accuracy=0.8817
k=11, Validation Accuracy=0.8804
k=12, Validation Accuracy=0.8758
k=13, Validation Accuracy=0.8754
k=14, Validation Accuracy=0.8684
k=15, Validation Accuracy=0.8648
k=16, Validation Accuracy=0.8634
k=17, Validation Accuracy=0.8591
k=18, Validation Accuracy=0.8576
k=19, Validation Accuracy=0.8552
k=20, Validation Accuracy=0.8519

Miglior k trovato: 1 con accuracy validation=0.9359
Test Accuracy con k=1: 0.6457

Classification report (test set):
              precision    recall  f1-score   support

           A       0.48      0.92      0.63       115
           B       0.99      1.00      1.00       115
           C       0.77      1.00      0.87       104
           D  

c:\Users\nicol\Desktop\Progetti\Machine learning\py311\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\nicol\Desktop\Progetti\Machine learning\py311\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\nicol\Desktop\Progetti\Machine learning\py311\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(ave

In [14]:
from sklearn.metrics import classification_report, accuracy_score

# ----- Valutazione finale sul test set -----
y_test_pred = final_knn.predict(X_test)

print("Classification Report (test set):")
print(classification_report(y_test, y_test_pred))

print("Test Accuracy:", accuracy_score(y_test, y_test_pred))

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
import joblib

# Dopo aver addestrato final_knn e scaler
joblib.dump(scaler, "scaler.pkl")
joblib.dump(final_knn, "knn_model.pkl")



Classification Report (test set):
              precision    recall  f1-score   support

           A       0.48      0.92      0.63       115
           B       0.99      1.00      1.00       115
           C       0.77      1.00      0.87       104
           D       0.91      0.25      0.39       115
           E       0.89      1.00      0.94       115
           F       1.00      1.00      1.00       115
           G       0.00      0.00      0.00       115
           H       0.63      1.00      0.77       115
           I       0.58      1.00      0.73       115
           J       1.00      0.31      0.48       115
           K       0.86      1.00      0.93       115
           L       0.77      0.98      0.86       115
           M       0.00      0.00      0.00       115
           N       0.98      0.40      0.57       114
           O       0.98      0.64      0.78        89
           P       0.29      0.80      0.43       115
           Q       0.77      0.30      0.43    

c:\Users\nicol\Desktop\Progetti\Machine learning\py311\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\nicol\Desktop\Progetti\Machine learning\py311\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\nicol\Desktop\Progetti\Machine learning\py311\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(ave

['knn_model.pkl']

In [12]:
import cv2
import mediapipe as mp
import numpy as np
from collections import deque, Counter
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
import joblib

# ----- Carica modello e scaler addestrati -----
scaler = joblib.load("scaler.pkl")
final_knn = joblib.load("knn_model.pkl")
labels = list(final_knn.classes_)

# ----- Buffer per stabilità predizione -----
prediction_buffer = deque(maxlen=10)
def get_stable_prediction(current_prediction):
    prediction_buffer.append(current_prediction)
    if len(prediction_buffer) < 5:
        return current_prediction
    counts = Counter(prediction_buffer)
    return counts.most_common(1)[0][0]

# ----- MediaPipe setup -----
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils

# Imposta webcam
cap = cv2.VideoCapture(0)

with mp_hands.Hands(
    max_num_hands=1,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5,
    static_image_mode=False
) as hands:

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            continue

        # Flip + RGB
        image = cv2.cvtColor(cv2.flip(frame, 1), cv2.COLOR_BGR2RGB)
        image.flags.writeable = False
        results = hands.process(image)
        image.flags.writeable = True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        if results.multi_hand_landmarks:
            for hand_landmarks in results.multi_hand_landmarks:
                # Disegna solo linee della mano
                mp_drawing.draw_landmarks(
                    image,
                    hand_landmarks,
                    mp_hands.HAND_CONNECTIONS,
                    landmark_drawing_spec=None,
                    connection_drawing_spec=mp_drawing.DrawingSpec(color=(0,255,0), thickness=2)
                )

                # Estrai keypoints x, y solo (42 valori)
                coords = np.array([[lm.x, lm.y] for lm in hand_landmarks.landmark]).flatten()

                # Trasforma con lo scaler
                coords_scaled = scaler.transform([coords])

                # Predizione KNN
                prediction = final_knn.predict(coords_scaled)[0]

                # Stabilizza predizione
                stable_prediction = get_stable_prediction(prediction)

                # Visualizza predizione
                cv2.rectangle(image, (0,0), (220,40), (245,90,16), -1)
                cv2.putText(image, f'Pred: {stable_prediction}', (10,25),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255,255,255), 2)

        else:
            prediction_buffer.clear()
            cv2.putText(image, "Hand not recognized", (10,25),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,0,255), 2)

        cv2.imshow('ASL Real-time', image)

        if cv2.waitKey(5) & 0xFF == 27:  # ESC
            break

cap.release()
cv2.destroyAllWindows()
